# Člověk v cyklu: Před-akční brány, třídění rizik a auditní záznamy

README k této lekci představuje člověka v cyklu krátkým úryvkem, který po agentovi žádá uživatele o `SCHVÁLENÍ` nebo `ZAMÍTNUTÍ` poté, co agent již vytvořil odpověď. Tento vzor je dobrým výchozím bodem, ale produkční implementace HITL běžně vyžadují tři další části:

1. **Před-akční bránu**, která běží **předtím**, než agent provede rizikový krok, aby se udržely náklady, nevratnost a latence pod kontrolou.
2. **Třídění rizik**, takže nízkorizikové akce se provedou automaticky, středně rizikové akce jsou schváleny v dávce a pouze vysoce rizikové akce vyžadují zásah člověka.
3. **Auditní záznam plus smyčka revize**, takže každé rozhodnutí brány je zaznamenáno jako JSONL a zamítnutí znovu vyvolá agenta s strukturovaným důvodem místo pouhého tisku `Revising...`.

Tento notebook staví každou z těchto částí na stejných primitivech jako `06-system-message-framework.ipynb`. Běží end-to-end v režimu `DEMO_MODE = True` (není potřeba interaktivní vstup) nebo s reálnými výzvami `input()`, když je `DEMO_MODE = False`. Poznámka: v DEMO_MODE je opakování třetího cíle skriptováno, takže mechanika smyčky je viditelná end-to-end. Reálné přeřazení založené na revizích vyžaduje `DEMO_MODE = False` a operátora.

**Mimo rozsah (řešeno v jiných lekcích):** autentizace a řízení přístupu (Lekce 06 README hrozba 2), middleware volání nástrojů (Lekce 14 MAF hloubková analýza), vzory debat mezi více agenty.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## Vzor 1: Brána před akcí

Úryvek HITL v README nejprve volá agenta, a pak žádá uživatele o schválení výstupu. To je **tok po akci**. Agent už provedl akci, takže náklady na volání LLM jsou již zaplaceny a jakýkoli vedlejší efekt (odeslaný e-mail, zapsaný řádek do databáze, zveřejněný komentář) již nastal.

**Tok před akcí** umisťuje bránu před tím, než agent provede riskantní krok. Agent navrhne akci, brána rozhodne, zda ji provést, a vedlejší efekt nastane pouze po schválení.

| Aspekt | Schválení po akci (úryvek z README) | Brána před akcí (v tomto notebooku) |
|---|---|---|
| Kdy probíhá schválení? | Po vytvoření výstupu agentem | Před provedením jakéhokoli vedlejšího efektu |
| Náklady na LLM při zamítnutí | Už zaplaceno | Zaplaceno pouze za návrh, nikoli za akci |
| Nevratné vedlejší efekty | Možné (akce už proběhla) | Zabráněno |
| Jasnost auditu | Schválení je výpis v konzoli | Schválení je záznam v JSON s časovou známkou, akcí, důvodem |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Pattern 2: Risk tiering

Not every action needs human approval. A read-only lookup against a public API has different stakes than sending a customer email. Treating both the same wastes operator attention and slows the agent.

A simple 3-tier model:

| Tier | Examples | Approval flow |
|---|---|---|
| `low` (read-only) | Search a knowledge base, look up flight options, fetch a public web page | Auto-execute, logged for audit |
| `medium` (cheap mutation) | Cache a result, increment a counter, schedule a reminder | Auto-execute, but batched daily review |
| `high` (external-facing or irreversible) | Send an email, charge a card, post to a public channel | Block on human approval |

This is one tiering. Production systems often use more granular tiers (e.g., AWS IAM permission levels, role-based access tiers). The 3-tier version below is the smallest useful version for an agent that mixes read-only and side-effecting actions.

The classifier below uses keyword heuristics so the demo stays deterministic and cheap. In a production system you would swap this for a learned classifier or a policy engine.


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Vzor 3: Auditní záznam a smyčka revizí

`print("Response approved.")` není auditní záznam. Pro důvěru by mělo být každé rozhodnutí brány zaznamenáno jako strukturovaná událost, kterou můžete později dotazovat, přehrát nebo připojit k přezkumu incidentu.

Dva díly:

1. **Pouze přidávaný JSONL.** Jeden řádek na rozhodnutí s časovou značkou, akcí, úrovní, rozhodnutím, důvodem. Snadné grepování, snadné pozdější odeslání do skutečného úložiště záznamů.
2. **Smyčka revizí při zamítnutí.** Když brána vrátí `deny`, agent se sám znovu vyzve s důvodem zamítnutí v kontextu, aby další návrh mohl problém vyřešit.


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Další zdroje

Několik dalších veřejných projektů implementuje varianty těchto vzorů HITL. Porovnejte přístupy, abyste našli, co vyhovuje vašemu stacku:

- **LangChain** nástroje s lidským zapojením ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): nástroje, které pozastaví provádění pro lidský vstup.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ toto restrukturalizoval): používá roli agenta specificky k reprezentaci člověka ve víceagentních konverzacích.
- **Microsoft Agent Framework (MAF)** middleware pro volání funkcí ([docs](https://learn.microsoft.com/agent-framework/)): middleware, který běží okolo každého volání nástroje/funkce, vhodný pro logiku schvalování a řízení přístupu.

Každý projekt řeší tyto tři podvzorové situace jinak: LangChain je zabalí jako nástroje, AutoGen používá roli agenta a Microsoft Agent Framework middleware pro volání funkcí. Přečtěte si jednu nebo dvě implementace od začátku do konce, než si vyberete návrh pro svého vlastního agenta.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Prohlášení o omezení odpovědnosti**:
Tento dokument byl přeložen pomocí AI překladatelské služby [Co-op Translator](https://github.com/Azure/co-op-translator). Přestože usilujeme o co největší přesnost, mějte prosím na paměti, že automatizované překlady mohou obsahovat chyby nebo nepřesnosti. Originální dokument v jeho mateřském jazyce by měl být považován za autoritativní zdroj. Pro kritické informace se doporučuje profesionální lidský překlad. Nejsme odpovědní za jakékoli nedorozumění nebo nesprávné interpretace vzniklé použitím tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
